# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [2]:
# Load the libraries as required.

import pandas as pd
data_path = "../../05_src/data/fires/forestfires.csv"
df = pd.read_csv(data_path)


In [3]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [4]:
X = df.drop(columns=['area'])
print("Features shape:", X.shape)

Features shape: (517, 12)


In [5]:
y = df['area']
print("Target shape:", y.shape)

Target shape: (517,)


# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = ['month', 'day']
num_cols = [c for c in X.columns if c not in cat_cols]

preproc1 = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='drop'
)

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [10]:
from sklearn.preprocessing import PowerTransformer
from sklearn.pipeline import Pipeline

cat_cols = ['month', 'day']
num_cols = [c for c in X.columns if c not in cat_cols]

preproc2 = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('transform', PowerTransformer(method='yeo-johnson')),  # non-linear
            ('scaler', StandardScaler())                           # scaling
        ]), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ],
    remainder='drop'
)

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [12]:
# Pipeline A = preproc1 + baseline

from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_validate
import numpy as np
import pandas as pd

pipeline_a = Pipeline([
    ("preprocessing", preproc1),
    ("regressor", Ridge(random_state=42))
])

param_grid_a = {
    "regressor__alpha": [0.1, 1.0, 5.0, 10.0, 25.0, 50.0]
}

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

grid_a = GridSearchCV(
    estimator=pipeline_a,
    param_grid=param_grid_a,
    scoring=scoring["rmse"],
    cv=5,
    n_jobs=-1,
    refit=True,
    verbose=0
)
grid_a.fit(X, y)

print("Pipeline A — best params:", grid_a.best_params_)
print("Pipeline A — best CV RMSE:", -grid_a.best_score_)

Pipeline A — best params: {'regressor__alpha': 50.0}
Pipeline A — best CV RMSE: 51.0207880370577


In [13]:
# Pipeline B = preproc2 + baseline

pipeline_b = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", Ridge(random_state=42))
])

param_grid_b = {
    "regressor__alpha": [0.1, 1.0, 5.0, 10.0, 25.0, 50.0]
}

scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

grid_b = GridSearchCV(
    estimator=pipeline_b,
    param_grid=param_grid_b,
    scoring=scoring["rmse"],
    cv=5,
    n_jobs=-1,
    refit=True,
    verbose=0
)
grid_b.fit(X, y)

best_b = grid_b.best_estimator_
print("Pipeline B — best params:", grid_b.best_params_)
print("Pipeline B — best CV RMSE:", -grid_b.best_score_)


Pipeline B — best params: {'regressor__alpha': 50.0}
Pipeline B — best CV RMSE: 51.02331267238149


In [14]:
# Pipeline C = preproc1 + advanced model

from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GridSearchCV

pipeline_c = Pipeline([
    ("preprocessing", preproc1),
    ("regressor", HistGradientBoostingRegressor(random_state=42))
])

param_grid_c = {
    "regressor__learning_rate": [0.05, 0.1],
    "regressor__max_depth": [None, 6]
}

grid_c = GridSearchCV(
    estimator=pipeline_c,
    param_grid=param_grid_c,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    refit=True
)

grid_c.fit(X, y)

print("Pipeline C — best parameters:", grid_c.best_params_)
print("Pipeline C — best CV RMSE:", -grid_c.best_score_)

Pipeline C — best parameters: {'regressor__learning_rate': 0.05, 'regressor__max_depth': 6}
Pipeline C — best CV RMSE: 56.41165368809963


In [ ]:
# Pipeline D = preproc2 + advanced model

pipeline_d = Pipeline([
    ("preprocessing", preproc2),
    ("regressor", HistGradientBoostingRegressor(random_state=42))
])

param_grid_d = {
    "regressor__learning_rate": [0.05, 0.1],
    "regressor__max_depth": [None, 6]
}

grid_d = GridSearchCV(
    estimator=pipeline_d,
    param_grid=param_grid_d,
    scoring="neg_root_mean_squared_error",
    cv=5,
    n_jobs=-1,
    refit=True
)

grid_d.fit(X, y)

print("Pipeline D — best parameters:", grid_d.best_params_)
print("Pipeline D — best CV RMSE:", -grid_d.best_score_)
    

Pipeline D — best parameters: {'regressor__learning_rate': 0.05, 'regressor__max_depth': 6}
Pipeline D — best CV RMSE: 56.41656092204122


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [25]:
from sklearn.model_selection import GridSearchCV
import numpy as np
import pandas as pd

pipelines = {
    "A (preproc1 + Ridge)": (pipeline_a, {"regressor__alpha": [0.1, 1, 10, 50]}),
    "B (preproc2 + Ridge)": (pipeline_b, {"regressor__alpha": [0.1, 1, 10, 50]}),
    "C (preproc1 + HGBR)":  (pipeline_c, {
        "regressor__learning_rate": [0.05, 0.1],
        "regressor__max_depth": [None, 6]
    }),
    "D (preproc2 + HGBR)":  (pipeline_d, {
        "regressor__learning_rate": [0.05, 0.1],
        "regressor__max_depth": [None, 6]
    })
}

results = []

for name, (pipe, grid) in pipelines.items():
    print(f"\n--- GridSearch for {name} ---")
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="neg_root_mean_squared_error",  # RMSE
        cv=5,
        n_jobs=-1,
        refit=True
    )
    gs.fit(X, y)

    results.append({
        "Pipeline": name,
        "Best Params": gs.best_params_,
        "CV RMSE (mean)": -gs.best_score_
    })

res_df = pd.DataFrame(results).sort_values("CV RMSE (mean)").reset_index(drop=True)
print("\n=== GridSearch Summary (sorted by RMSE) ===")
print(res_df.to_string(index=False))



--- GridSearch for A (preproc1 + Ridge) ---

--- GridSearch for B (preproc2 + Ridge) ---

--- GridSearch for C (preproc1 + HGBR) ---

--- GridSearch for D (preproc2 + HGBR) ---

=== GridSearch Summary (sorted by RMSE) ===
            Pipeline                                                   Best Params  CV RMSE (mean)
A (preproc1 + Ridge)                                      {'regressor__alpha': 50}       51.020788
B (preproc2 + Ridge)                                      {'regressor__alpha': 50}       51.023313
 C (preproc1 + HGBR) {'regressor__learning_rate': 0.05, 'regressor__max_depth': 6}       56.411654
 D (preproc2 + HGBR) {'regressor__learning_rate': 0.05, 'regressor__max_depth': 6}       56.416561


# Evaluate

+ Which model has the best performance?

In [ ]:
#model c - model a & b dont capture the non linearities 

# Export

+ Save the best performing model to a pickle file.

In [27]:
import pickle


best_model = grid_c.best_estimator_   


with open("best_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("✅ Best model saved to best_model.pkl")

✅ Best model saved to best_model.pkl


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
#For a single test observation, temperature, humidity, and wind were the most influential features.
#Across the dataset, weather-related features dominate global importance, while rain, day, and spatial coordinates are least useful.
#I would test removing these weaker features by retraining and comparing performance. If RMSE does not increase, it suggests these features do not enhance predictive power.


*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.